## Set Up

In [ ]:
# CUDA allocator tuning (set before importing torch in setup cells).
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:256')
print('PYTORCH_CUDA_ALLOC_CONF =', os.environ.get('PYTORCH_CUDA_ALLOC_CONF'))

In [ ]:
# Colab/runtime setup (aligned with old eval notebook repo loading)
import os
import sys
from pathlib import Path

import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Mirror old notebook behavior: in Colab, use secrets-driven auth.
IN_COLAB = 'google.colab' in sys.modules
userdata = None
if IN_COLAB:
    try:
        from google.colab import userdata as _colab_userdata
        userdata = _colab_userdata
    except Exception as exc:
        raise RuntimeError('Colab secrets API is unavailable; cannot continue.') from exc

    REPO_URL = 'github.com/Ancorro/AI535project.git'
    REPO_DIR = '/content/AI535project'
    TARGET_BRANCH = 'V5'

    # GitHub secret from Colab secrets only.
    try:
        gh_token = userdata.get('GITHUB_TOKEN') or userdata.get('GH_TOKEN')
    except Exception:
        gh_token = None
    if not gh_token:
        raise RuntimeError('Missing Colab secret GITHUB_TOKEN (or GH_TOKEN); refusing to continue.')

    auth_url = f'https://{gh_token}@{REPO_URL}'
    if os.path.isdir(REPO_DIR):
        print(f'Repo exists, updating branch {TARGET_BRANCH} ...')
        os.system(f'git -C {REPO_DIR} fetch origin {TARGET_BRANCH}')
        os.system(f'git -C {REPO_DIR} checkout {TARGET_BRANCH}')
        os.system(f'git -C {REPO_DIR} pull origin {TARGET_BRANCH}')
    else:
        print(f'Repo not found, cloning branch {TARGET_BRANCH} ...')
        os.system(f'git clone --branch {TARGET_BRANCH} --single-branch {auth_url} {REPO_DIR}')
    del auth_url

    # Hugging Face secret from Colab secrets only.
    try:
        hf_token = userdata.get('HF_TOKEN') or userdata.get('HUGGINGFACE_TOKEN')
    except Exception:
        hf_token = None
    if not hf_token:
        raise RuntimeError('Missing Colab secret HF_TOKEN (or HUGGINGFACE_TOKEN); refusing to continue.')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGINGFACE_TOKEN'] = hf_token

# Resolve repository root robustly (works for Colab and local VS Code).
cwd = Path.cwd().resolve()
repo_candidate = None
for p in [cwd, *cwd.parents]:
    if (p / 'train' / 'train.py').exists() and (p / 'logic').exists():
        repo_candidate = p
        break
if repo_candidate is None and IN_COLAB and Path('/content/AI535project').exists():
    repo_candidate = Path('/content/AI535project').resolve()
if repo_candidate is None:
    raise FileNotFoundError('Could not locate repo root containing train/train.py and logic/.')

REPO_ROOT = repo_candidate
PROJECT_ROOT = REPO_ROOT  # backwards-compatible name used in this notebook
print('REPO_ROOT:', REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Enforce W&B setup from Colab secret. Do not continue on failure.
try:
    import wandb
except Exception as exc:
    raise RuntimeError('wandb is required but not installed; install it before continuing.') from exc

if IN_COLAB:
    try:
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None
    if not wandb_key:
        raise RuntimeError('Missing Colab secret WANDB_API_KEY; refusing to continue.')
    try:
        wandb.login(key=wandb_key, relogin=True)
    except Exception as exc:
        raise RuntimeError('W&B login failed with WANDB_API_KEY secret; refusing to continue.') from exc
else:
    # Local fallback for non-Colab runs.
    wandb_key = os.getenv('WANDB_API_KEY')
    if not wandb_key:
        raise RuntimeError('Missing WANDB_API_KEY env var for local run; refusing to continue.')
    try:
        wandb.login(key=wandb_key, relogin=True)
    except Exception as exc:
        raise RuntimeError('W&B login failed with local WANDB_API_KEY; refusing to continue.') from exc

print('W&B authenticated successfully.')
if IN_COLAB:
    print('GitHub and Hugging Face tokens loaded from Colab secrets.')

# Optional installs for fresh Colab runtimes:
# !pip install -q transformers datasets accelerate wandb

## Parameters

In [ ]:
# Experiment config
from dataclasses import dataclass

@dataclass
class ExpConfig:
    model_name: str = 'meta-llama/Llama-3.1-8B'
    run_model: str = 'logic_llama'  # choose one: 'baseline_llama' or 'logic_llama'
    run_seeds: tuple[int, ...] = (42,)  # run these seeds back-to-back

    # Dataset config (set dataset_name to 'nyu-mll/multi_nli' or 'proofwriter').
    dataset_name: str = 'proofwriter'
    mnli_train_split: str = 'train'
    mnli_val_split: str = 'validation_matched'  # or 'validation_mismatched'

    # ProofWriter config (used when dataset_name='proofwriter').
    proofwriter_config: str = 'default'
    proofwriter_depth: str = 'all'
    proofwriter_train_split: str = 'train'
    proofwriter_val_split: str = 'validation'

    # Memory-safe defaults for 8B full-finetune workflows.
    max_length: int = 0  # 0 disables cap; 256 is much safer for VRAM
    train_batch_size: int = 30
    eval_batch_size: int = 30
    train_max_samples: int = 10000
    val_max_samples: int = 5000
    k_epochs: int = 2
    eval_checks_per_epoch: int = 4  # number of mini-eval checks per epoch
    mini_eval_batches_per_check: int = 2  # batches per mini-eval check
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    seed: int = 42
    logic_dim: int = 256
    num_gates: int = 8
    cross_attn_heads: int = 4
    alpha_init: float = 0.1
    learn_fusion_alpha: bool = True  # if False, fusion alpha is fixed at alpha_init
    gate_window_axis: str = 'inter_token'  # choose one: 'inter_token' or 'intra_token'
    gate_mode: str = 'ste_binary'  # choose one: 'soft' or 'ste_binary'
    use_param_matched_baseline: bool = True  # if True, baseline_llama uses no-gate control with matched layer capacity
    param_matched_no_gate_update_type: str = 'mlp'  # one of: 'mlp', 'linear'
    use_bf16_if_available: bool = True
    enable_gradient_checkpointing: bool = False
    disable_kv_cache_during_train: bool = True
    wandb_project: str = 'ai535-logic-vs-llama'
    wandb_group: str = 'v5-eval-mnli'

CFG = ExpConfig()
if CFG.gate_window_axis not in {'inter_token', 'intra_token'}:
    raise ValueError("CFG.gate_window_axis must be one of {'inter_token', 'intra_token'}")
if CFG.gate_mode not in {'soft', 'ste_binary'}:
    raise ValueError("CFG.gate_mode must be one of {'soft', 'ste_binary'}")
print(CFG)

In [ ]:
# Imports + reproducibility + device
import gc
import importlib
import inspect
import math
import random
import time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Force-reload local modules so notebook always uses current on-disk code.
import logic.core.fusion as fusion_mod
import logic.core.logic_llama_model as logic_llama_mod
import logic.core.baseline_model as baseline_mod
import logic.core.data_utils as data_utils_mod
for _mod in (fusion_mod, logic_llama_mod, baseline_mod, data_utils_mod):
    importlib.reload(_mod)

from logic.core.fusion import FusionMLP, LinearFusionBridge
from logic.core.baseline_model import BaselineModel
from logic.core.logic_llama_model import LogicLlamaModel
from logic.core.data_utils import load_proofwriter

def print_runtime_code_audit() -> None:
    targets = [
        ('LogicLlamaModel', LogicLlamaModel.__init__),
        ('BaselineModel', BaselineModel.__init__),
        ('FusionMLP', FusionMLP.__init__),
        ('LinearFusionBridge', LinearFusionBridge.__init__),
    ]
    print('Runtime code audit:')
    for name, fn in targets:
        file_path = inspect.getsourcefile(fn) or inspect.getfile(fn)
        line_no = inspect.getsourcelines(fn)[1]
        print(f'  {name}: {file_path}:{line_no}')

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

use_bf16 = bool(CFG.use_bf16_if_available and torch.cuda.is_available() and torch.cuda.is_bf16_supported())
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print('AMP enabled:', torch.cuda.is_available(), 'dtype:', amp_dtype)

scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available() and not use_bf16)

print_runtime_code_audit()

In [ ]:
# Data loaders (MultiNLI or ProofWriter)
from datasets import load_dataset

dataset_key = str(CFG.dataset_name).strip().lower()
tokenizer = getattr(data_utils_mod, 'get_tokenizer')(CFG.model_name)

# max_length=0 means no truncation cap for MNLI path; ProofWriter loader needs an int.
effective_max_length = None if int(CFG.max_length) == 0 else int(CFG.max_length)
pw_max_length = 256 if int(CFG.max_length) == 0 else int(CFG.max_length)

if dataset_key in {'proofwriter', 'tasksource/proofwriter', 'longface/proofwriter'}:
    proofwriter_cfg = str(getattr(CFG, 'proofwriter_config', 'default')).strip()
    if proofwriter_cfg.lower() in {'', 'none', 'null'}:
        proofwriter_cfg = None

    train_ds, train_collate = load_proofwriter(
        model_name=CFG.model_name,
        max_length=pw_max_length,
        depth=CFG.proofwriter_depth,
        split=CFG.proofwriter_train_split,
        config_name=proofwriter_cfg,
        max_samples=CFG.train_max_samples,
        seed=int(CFG.seed),
    )
    val_ds, val_collate = load_proofwriter(
        model_name=CFG.model_name,
        max_length=pw_max_length,
        depth=CFG.proofwriter_depth,
        split=CFG.proofwriter_val_split,
        config_name=proofwriter_cfg,
        max_samples=CFG.val_max_samples,
        seed=int(CFG.seed),
    )

    train_loader = DataLoader(train_ds, batch_size=CFG.train_batch_size, shuffle=True, collate_fn=train_collate)
    val_loader = DataLoader(val_ds, batch_size=CFG.eval_batch_size, shuffle=False, collate_fn=val_collate)

    print('Dataset: proofwriter')
    print('Config:', proofwriter_cfg, 'Depth:', CFG.proofwriter_depth)
    print('Splits:', CFG.proofwriter_train_split, '(train),', CFG.proofwriter_val_split, '(val)')
    print('Effective max_length:', pw_max_length)
    print('Train samples:', len(train_ds), 'Val samples:', len(val_ds))
    print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))

else:
    mnli = load_dataset(CFG.dataset_name)
    if CFG.mnli_train_split not in mnli or CFG.mnli_val_split not in mnli:
        raise ValueError(
            f"Requested splits not found in {CFG.dataset_name}. "
            f"Available: {sorted(mnli.keys())}"
        )

    train_ds = mnli[CFG.mnli_train_split]
    val_ds = mnli[CFG.mnli_val_split]

    # Deterministic sample caps (match existing notebook behavior).
    if CFG.train_max_samples is not None and int(CFG.train_max_samples) > 0 and len(train_ds) > int(CFG.train_max_samples):
        train_ds = train_ds.shuffle(seed=int(CFG.seed)).select(range(int(CFG.train_max_samples)))
    if CFG.val_max_samples is not None and int(CFG.val_max_samples) > 0 and len(val_ds) > int(CFG.val_max_samples):
        val_ds = val_ds.shuffle(seed=int(CFG.seed)).select(range(int(CFG.val_max_samples)))

    # MultiNLI labels: 0=entailment, 1=neutral, 2=contradiction.
    # ProofWriter-style labels used in this repo: 0=contradiction/false, 1=entailment/true, 2=unknown/neutral.
    MNLI_TO_PROOFWRITER = {0: 1, 1: 2, 2: 0}

    def collate_fn(examples):
        premises = [str(ex['premise']) for ex in examples]
        hypotheses = [str(ex['hypothesis']) for ex in examples]
        raw_labels = [int(ex['label']) for ex in examples]

        # Filter out examples with invalid labels (e.g., -1) before tensorization.
        keep = [i for i, y in enumerate(raw_labels) if y in MNLI_TO_PROOFWRITER]
        if not keep:
            raise RuntimeError('Batch contained no valid MNLI labels after filtering.')

        premises = [premises[i] for i in keep]
        hypotheses = [hypotheses[i] for i in keep]
        labels = torch.tensor([MNLI_TO_PROOFWRITER[raw_labels[i]] for i in keep], dtype=torch.long)

        batch = tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=effective_max_length,
            return_tensors='pt',
        )
        batch['labels'] = labels
        return dict(batch)

    train_loader = DataLoader(train_ds, batch_size=CFG.train_batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=CFG.eval_batch_size, shuffle=False, collate_fn=collate_fn)

    print('Dataset:', CFG.dataset_name)
    print('Splits:', CFG.mnli_train_split, '(train),', CFG.mnli_val_split, '(val)')
    print('Label map MNLI->ProofWriter:', MNLI_TO_PROOFWRITER)
    print('Effective max_length:', effective_max_length)
    print('Train samples:', len(train_ds), 'Val samples:', len(val_ds))
    print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))

In [ ]:
# Build models (same head shape via num_labels=3)
NUM_LABELS = 3
VALID_MODELS = {'baseline_llama', 'logic_llama'}

def _apply_memory_optimizations(model: torch.nn.Module) -> None:
    """Enable memory-saving flags on any compatible backbone/module."""
    if not bool(getattr(CFG, 'enable_gradient_checkpointing', True)):
        return

    candidates = [model]
    for attr in ('llama', 'model', 'backbone', 'base_model'):
        sub = getattr(model, attr, None)
        if sub is not None:
            candidates.append(sub)

    seen = set()
    for m in candidates:
        if id(m) in seen:
            continue
        seen.add(id(m))

        if bool(getattr(CFG, 'disable_kv_cache_during_train', True)):
            cfg = getattr(m, 'config', None)
            if cfg is not None and hasattr(cfg, 'use_cache'):
                cfg.use_cache = False

        fn = getattr(m, 'gradient_checkpointing_enable', None)
        if callable(fn):
            try:
                fn()
            except Exception:
                pass

def build_baseline() -> torch.nn.Module:
    if CFG.use_param_matched_baseline:
        # Matched baseline path: no-gate stream (no cross-attention), H->L projection per layer, MLP fusion.
        model = LogicLlamaModel(
            model_name=CFG.model_name,
            stream_logic_during_backbone=True,
            logic_dim=CFG.logic_dim,
            num_gates=CFG.num_gates,
            cross_attn_heads=CFG.cross_attn_heads,
            gate_window_axis=CFG.gate_window_axis,
            gate_mode=CFG.gate_mode,
            use_no_gate_stream=True,
            no_gate_update_type=CFG.param_matched_no_gate_update_type,
            no_gate_match_logic_params=True,
            routing_top_k=None,
            routing_train_mode='dense',
            routing_train_cutoff=0.0,
            fusion_mode='mlp',
            alpha_init=CFG.alpha_init,
            learn_fusion_alpha=CFG.learn_fusion_alpha,
            num_labels=NUM_LABELS,
        )
        _apply_memory_optimizations(model)
        return model.to(device)

    model = BaselineModel(
        model_name=CFG.model_name,
        num_labels=NUM_LABELS,
    )
    _apply_memory_optimizations(model)
    return model.to(device)


def build_logic() -> torch.nn.Module:
    model = LogicLlamaModel(
        model_name=CFG.model_name,
        stream_logic_during_backbone=True,
        logic_dim=CFG.logic_dim,
        num_gates=CFG.num_gates,
        cross_attn_heads=CFG.cross_attn_heads,
        gate_window_axis=CFG.gate_window_axis,
        gate_mode=CFG.gate_mode,
        routing_top_k=None,
        routing_train_mode='dense',
        routing_train_cutoff=0.0,
        fusion_mode='linear_bridge',
        alpha_init=CFG.alpha_init,
        learn_fusion_alpha=CFG.learn_fusion_alpha,
        num_labels=NUM_LABELS,
    )
    _apply_memory_optimizations(model)
    return model.to(device)


def build_model(model_name: str) -> tuple[str, torch.nn.Module]:
    model_name = model_name.strip().lower()
    if model_name not in VALID_MODELS:
        raise ValueError(f"model_name must be one of {sorted(VALID_MODELS)}, got '{model_name}'")
    if model_name == 'baseline_llama':
        return model_name, build_baseline()
    return model_name, build_logic()


selected_model_name = CFG.run_model.strip().lower()
if selected_model_name not in VALID_MODELS:
    raise ValueError(f"CFG.run_model must be one of {sorted(VALID_MODELS)}, got '{CFG.run_model}'")

print('Planned single-model run:', selected_model_name)
print('Gate comparison axis:', CFG.gate_window_axis)
print('Gate mode:', CFG.gate_mode)
print('Learn fusion alpha:', CFG.learn_fusion_alpha)
print('Gradient checkpointing:', CFG.enable_gradient_checkpointing)
print('Disable KV cache during train:', CFG.disable_kv_cache_during_train)
print_runtime_code_audit()
if selected_model_name == 'baseline_llama' and CFG.use_param_matched_baseline:
    print('Baseline mode: parameter-matched no-gate control is ENABLED (cross-attention is DISABLED)')

In [ ]:
# Train/eval helpers
def _safe_float(value, default: float = float('nan')) -> float:
    try:
        x = float(value)
        return x if math.isfinite(x) else default
    except Exception:
        return default


def _compute_grad_norm(model: torch.nn.Module) -> float:
    sq_sum = 0.0
    found = False
    for p in model.parameters():
        if p.grad is None:
            continue
        grad = p.grad.detach()
        sq_sum += float(torch.sum(grad * grad).item())
        found = True
    if not found:
        return 0.0
    return float(math.sqrt(max(sq_sum, 0.0)))


def _count_trainable_params(model: torch.nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def _get_fusion_alpha_param(model: torch.nn.Module) -> torch.nn.Parameter | None:
    fusion = getattr(model, 'fusion', None)
    if fusion is None:
        return None
    alpha = getattr(fusion, 'alpha', None)
    if isinstance(alpha, torch.nn.Parameter):
        return alpha
    return None


def _build_optimizer(
    model: torch.nn.Module,
    base_lr: float,
    weight_decay: float,
    fusion_alpha_lr_mult: float = 20.0,
) -> tuple[torch.optim.Optimizer, bool]:
    alpha_param = _get_fusion_alpha_param(model)
    alpha_param_id = id(alpha_param) if alpha_param is not None else None

    main_params: list[torch.nn.Parameter] = []
    alpha_params: list[torch.nn.Parameter] = []
    for p in model.parameters():
        if not p.requires_grad:
            continue
        if alpha_param_id is not None and id(p) == alpha_param_id:
            alpha_params.append(p)
        else:
            main_params.append(p)

    param_groups = []
    if main_params:
        param_groups.append({
            'params': main_params,
            'lr': float(base_lr),
            'weight_decay': float(weight_decay),
        })

    alpha_trainable = bool(alpha_params)
    if alpha_trainable:
        alpha_lr = float(base_lr) * float(max(fusion_alpha_lr_mult, 1.0))
        param_groups.append({
            'params': alpha_params,
            'lr': alpha_lr,
            'weight_decay': 0.0,
        })

    if not param_groups:
        raise RuntimeError('No trainable parameters found; cannot build optimizer.')

    optimizer = torch.optim.AdamW(param_groups)
    return optimizer, alpha_trainable


def _read_gpu_power_w() -> float | None:
    if not torch.cuda.is_available():
        return None
    # Try NVML first; fallback to nvidia-smi query if NVML bindings are unavailable.
    try:
        import pynvml  # type: ignore
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        power_mw = pynvml.nvmlDeviceGetPowerUsage(handle)
        return float(power_mw) / 1000.0
    except Exception:
        pass
    try:
        import subprocess
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip().splitlines()
        if not out:
            return None
        return float(out[0].strip())
    except Exception:
        return None


def _extract_routing_entropy(outputs) -> float | None:
    history = getattr(outputs, 'routing_history', None)
    if not history:
        return None
    entropies = []
    for weights in history:
        if not isinstance(weights, torch.Tensor) or weights.numel() == 0:
            continue
        probs = weights.detach().float()
        probs = probs.clamp_min(1e-12)
        probs = probs / probs.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        entropy = -(probs * probs.log()).sum(dim=-1).mean()
        if torch.isfinite(entropy):
            entropies.append(float(entropy.item()))
    if not entropies:
        return None
    return float(sum(entropies) / len(entropies))


def _extract_fusion_alpha(model: torch.nn.Module) -> float | None:
    fusion = getattr(model, 'fusion', None)
    if fusion is None:
        return None
    alpha = getattr(fusion, 'alpha', None)
    if alpha is None:
        return None
    if isinstance(alpha, torch.Tensor):
        alpha = alpha.detach().float().mean().item()
    return _safe_float(alpha, default=float('nan'))


def _fusion_alpha_trainable(model: torch.nn.Module) -> bool | None:
    fusion = getattr(model, 'fusion', None)
    if fusion is None:
        return None
    alpha = getattr(fusion, 'alpha', None)
    if isinstance(alpha, torch.nn.Parameter):
        return bool(alpha.requires_grad)
    return False


def _aggregate_checks(values: list[float]) -> dict:
    arr = np.asarray(values, dtype=np.float64)
    return {
        'mean': float(np.mean(arr)),
        'std': float(np.std(arr)),
        'min': float(np.min(arr)),
        'max': float(np.max(arr)),
    }


@torch.no_grad()
def evaluate_model(model: torch.nn.Module, loader: DataLoader, max_batches: int | None = None) -> dict:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    routing_entropy_vals: list[float] = []

    for batch_idx, batch in enumerate(loader):
        if max_batches is not None and batch_idx >= int(max_batches):
            break

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=torch.cuda.is_available()):
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = out.logits
            loss = F.cross_entropy(logits, labels)

        total_loss += float(loss.item()) * input_ids.size(0)
        preds = logits.argmax(dim=-1)
        total_correct += int((preds == labels).sum().item())
        total_count += int(input_ids.size(0))

        entropy = _extract_routing_entropy(out)
        if entropy is not None and math.isfinite(entropy):
            routing_entropy_vals.append(float(entropy))

    return {
        'loss': total_loss / max(total_count, 1),
        'acc': total_correct / max(total_count, 1),
        'routing_entropy': (sum(routing_entropy_vals) / len(routing_entropy_vals)) if routing_entropy_vals else None,
    }


def train_one_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    trainable_params: int,
    train_flop_factor: float = 6.0,
) -> dict:
    model.train()
    total_loss = 0.0
    total_count = 0
    total_tokens = 0
    total_grad_norm = 0.0
    grad_steps = 0
    non_finite_loss_count = 0
    routing_entropy_vals: list[float] = []
    alpha_grad_abs_means: list[float] = []
    energy_gpu_wh = 0.0
    power_samples: list[float] = []

    epoch_start = time.perf_counter()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    alpha_param = _get_fusion_alpha_param(model)

    for batch in loader:
        batch_start = time.perf_counter()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=torch.cuda.is_available()):
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = out.logits
            loss = F.cross_entropy(logits, labels)

        if not torch.isfinite(loss):
            non_finite_loss_count += 1
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = _compute_grad_norm(model)
        if alpha_param is not None and alpha_param.grad is not None:
            alpha_grad_abs_means.append(float(alpha_param.grad.detach().abs().mean().item()))
        scaler.step(optimizer)
        scaler.update()

        entropy = _extract_routing_entropy(out)
        if entropy is not None and math.isfinite(entropy):
            routing_entropy_vals.append(float(entropy))

        total_grad_norm += float(grad_norm)
        grad_steps += 1
        batch_size = int(input_ids.size(0))
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size
        total_tokens += int(attention_mask.sum().item())

        batch_sec = max(time.perf_counter() - batch_start, 1e-9)
        power_w = _read_gpu_power_w()
        if power_w is not None and math.isfinite(power_w) and power_w >= 0.0:
            power_samples.append(float(power_w))
            energy_gpu_wh += float(power_w) * batch_sec / 3600.0

    epoch_sec = max(time.perf_counter() - epoch_start, 1e-9)
    peak_gpu_mem_gb = (torch.cuda.max_memory_allocated() / (1024 ** 3)) if torch.cuda.is_available() else 0.0
    avg_grad_norm = total_grad_norm / max(grad_steps, 1)
    gpu_avg_power_w = (sum(power_samples) / len(power_samples)) if power_samples else None
    alpha_grad_abs_mean = (sum(alpha_grad_abs_means) / len(alpha_grad_abs_means)) if alpha_grad_abs_means else None

    # FLOPs estimate: ~6 * trainable_params * tokens for full training step accounting.
    flops_per_token_est = float(train_flop_factor * trainable_params)
    flops_per_epoch_est = float(flops_per_token_est * total_tokens)
    flops_per_step_est = float(flops_per_epoch_est / max(grad_steps, 1))

    return {
        'loss': total_loss / max(total_count, 1),
        'grad_norm': avg_grad_norm,
        'alpha_grad_abs_mean': alpha_grad_abs_mean,
        'non_finite_loss_count': int(non_finite_loss_count),
        'samples_per_sec': total_count / epoch_sec,
        'tokens_per_sec': total_tokens / epoch_sec,
        'epoch_sec': epoch_sec,
        'peak_gpu_mem_gb': float(peak_gpu_mem_gb),
        'routing_entropy': (sum(routing_entropy_vals) / len(routing_entropy_vals)) if routing_entropy_vals else None,
        'flops_per_token_est': flops_per_token_est,
        'flops_per_step_est': flops_per_step_est,
        'flops_per_epoch_est': flops_per_epoch_est,
        'energy_gpu_wh': float(energy_gpu_wh),
        'gpu_avg_power_w': gpu_avg_power_w,
    }


def run_training(model: torch.nn.Module, name: str, epochs: int, wandb_run=None) -> list[dict]:
    alpha_lr_mult = float(getattr(CFG, 'fusion_alpha_lr_mult', 20.0))
    optimizer, alpha_in_optimizer = _build_optimizer(
        model,
        base_lr=CFG.learning_rate,
        weight_decay=CFG.weight_decay,
        fusion_alpha_lr_mult=alpha_lr_mult,
    )
    history = []
    train_start = time.perf_counter()
    trainable_params = _count_trainable_params(model)
    cumulative_flops_est = 0.0
    cumulative_energy_gpu_wh = 0.0
    eval_checks = max(1, int(CFG.eval_checks_per_epoch))
    mini_eval_batches = max(1, int(CFG.mini_eval_batches_per_check))

    fusion_alpha_trainable = _fusion_alpha_trainable(model)
    if CFG.learn_fusion_alpha and not alpha_in_optimizer:
        raise RuntimeError(
            'CFG.learn_fusion_alpha=True but fusion alpha is not in optimizer parameter groups. '
            'Re-run model build cells to refresh notebook state.'
        )
    print(
        'Optimizer fusion-alpha setup:',
        {
            'cfg_learn_fusion_alpha': bool(CFG.learn_fusion_alpha),
            'alpha_trainable': fusion_alpha_trainable,
            'alpha_in_optimizer': alpha_in_optimizer,
            'fusion_alpha_lr_mult': alpha_lr_mult if alpha_in_optimizer else None,
        },
    )

    init_val = evaluate_model(model, val_loader)
    init_alpha = _extract_fusion_alpha(model)
    init_row = {
        'epoch': 0,
        'train_loss': None,
        'val_loss': init_val['loss'],
        'val_acc': init_val['acc'],
        'routing_entropy': init_val['routing_entropy'],
        'fusion_alpha': init_alpha,
        'fusion_alpha_delta': 0.0,
        'fusion_alpha_grad_abs_mean': None,
        'flops_per_token_est': float(6.0 * trainable_params),
        'flops_per_step_est': None,
        'flops_per_epoch_est': 0.0,
        'energy_gpu_wh': 0.0,
        'gpu_avg_power_w': None,
        'val_checks_per_epoch': eval_checks,
        'mini_eval_batches_per_check': mini_eval_batches,
        'val_loss_std': 0.0,
        'val_loss_min': init_val['loss'],
        'val_loss_max': init_val['loss'],
        'val_acc_std': 0.0,
        'val_acc_min': init_val['acc'],
        'val_acc_max': init_val['acc'],
        'val_loss_mini_mean': init_val['loss'],
        'val_acc_mini_mean': init_val['acc'],
    }
    history.append(init_row)
    prev_alpha = init_alpha
    print(f'[{name}] epoch=0 val_loss={init_val["loss"]:.4f} val_acc={init_val["acc"]:.4f} fusion_alpha={_safe_float(init_alpha):.6f}')
    if wandb_run is not None:
        wandb_run.log({
            'epoch': 0,
            'train/loss': float('nan'),
            'val/loss': _safe_float(init_val['loss']),
            'val/acc': _safe_float(init_val['acc']),
            'val/loss_std': 0.0,
            'val/loss_min': _safe_float(init_val['loss']),
            'val/loss_max': _safe_float(init_val['loss']),
            'val/acc_std': 0.0,
            'val/acc_min': _safe_float(init_val['acc']),
            'val/acc_max': _safe_float(init_val['acc']),
            'val/loss_mini_mean': _safe_float(init_val['loss']),
            'val/acc_mini_mean': _safe_float(init_val['acc']),
            'eval/checks_per_epoch': float(eval_checks),
            'eval/mini_eval_batches_per_check': float(mini_eval_batches),
            'train/routing_entropy': _safe_float(init_val['routing_entropy']),
            'train/fusion_alpha': _safe_float(init_row['fusion_alpha']),
            'train/fusion_alpha_delta': 0.0,
            'train/fusion_alpha_grad_abs_mean': float('nan'),
            'val/lr': _safe_float(optimizer.param_groups[0]['lr']),
            'reliability/non_finite_loss_count': 0.0,
            'perf/samples_per_sec': float('nan'),
            'perf/tokens_per_sec': float('nan'),
            'time/epoch_sec': 0.0,
            'system/peak_gpu_mem_gb': 0.0,
            'perf/flops_per_token_est': _safe_float(init_row['flops_per_token_est']),
            'perf/flops_per_step_est': float('nan'),
            'perf/flops_per_epoch_est': 0.0,
            'energy/gpu_wh': 0.0,
            'energy/gpu_avg_power_w': float('nan'),
        }, step=0)

    for epoch in range(1, epochs + 1):
        train_stats = train_one_epoch(
            model,
            train_loader,
            optimizer,
            trainable_params=trainable_params,
            train_flop_factor=6.0,
        )

        mini_val_loader = DataLoader(
            val_ds,
            batch_size=CFG.eval_batch_size,
            shuffle=True,
            collate_fn=collate_fn,
        )
        mini_checks_rows = [
            evaluate_model(model, mini_val_loader, max_batches=mini_eval_batches)
            for _ in range(eval_checks)
        ]
        mini_loss_stats = _aggregate_checks([row['loss'] for row in mini_checks_rows])
        mini_acc_stats = _aggregate_checks([row['acc'] for row in mini_checks_rows])

        full_val = evaluate_model(model, val_loader)

        routing_candidates = [row['routing_entropy'] for row in mini_checks_rows if row['routing_entropy'] is not None]
        if full_val['routing_entropy'] is not None:
            routing_candidates.append(full_val['routing_entropy'])
        val_routing_entropy = float(np.mean(routing_candidates)) if routing_candidates else None

        fusion_alpha = _extract_fusion_alpha(model)
        fusion_alpha_delta = (
            float(fusion_alpha - prev_alpha)
            if fusion_alpha is not None and prev_alpha is not None and math.isfinite(float(fusion_alpha)) and math.isfinite(float(prev_alpha))
            else float('nan')
        )
        prev_alpha = fusion_alpha
        cumulative_flops_est += float(train_stats['flops_per_epoch_est'])
        cumulative_energy_gpu_wh += float(train_stats['energy_gpu_wh'])
        row = {
            'epoch': epoch,
            'train_loss': train_stats['loss'],
            'val_loss': full_val['loss'],
            'val_acc': full_val['acc'],
            'val_loss_std': mini_loss_stats['std'],
            'val_loss_min': mini_loss_stats['min'],
            'val_loss_max': mini_loss_stats['max'],
            'val_acc_std': mini_acc_stats['std'],
            'val_acc_min': mini_acc_stats['min'],
            'val_acc_max': mini_acc_stats['max'],
            'val_loss_mini_mean': mini_loss_stats['mean'],
            'val_acc_mini_mean': mini_acc_stats['mean'],
            'val_checks_per_epoch': eval_checks,
            'mini_eval_batches_per_check': mini_eval_batches,
            'routing_entropy': val_routing_entropy if val_routing_entropy is not None else train_stats['routing_entropy'],
            'fusion_alpha': fusion_alpha,
            'fusion_alpha_delta': fusion_alpha_delta,
            'fusion_alpha_grad_abs_mean': train_stats['alpha_grad_abs_mean'],
            'grad_norm': train_stats['grad_norm'],
            'lr': optimizer.param_groups[0]['lr'],
            'non_finite_loss_count': train_stats['non_finite_loss_count'],
            'samples_per_sec': train_stats['samples_per_sec'],
            'tokens_per_sec': train_stats['tokens_per_sec'],
            'epoch_sec': train_stats['epoch_sec'],
            'peak_gpu_mem_gb': train_stats['peak_gpu_mem_gb'],
            'flops_per_token_est': train_stats['flops_per_token_est'],
            'flops_per_step_est': train_stats['flops_per_step_est'],
            'flops_per_epoch_est': train_stats['flops_per_epoch_est'],
            'energy_gpu_wh': train_stats['energy_gpu_wh'],
            'gpu_avg_power_w': train_stats['gpu_avg_power_w'],
        }
        history.append(row)
        print(
            f'[{name}] epoch={epoch} '
            f'train_loss={row["train_loss"]:.4f} '
            f'val_loss(full)={row["val_loss"]:.4f} '
            f'val_acc(full)={row["val_acc"]:.4f} '
            f'fusion_alpha={_safe_float(row["fusion_alpha"]):.6f} '
            f'delta={_safe_float(row["fusion_alpha_delta"]):.6e} '
            f'alpha_grad_abs={_safe_float(row["fusion_alpha_grad_abs_mean"]):.3e}'
        )
        if wandb_run is not None:
            wandb_run.log({
                'epoch': epoch,
                'train/loss': _safe_float(row['train_loss']),
                'val/loss': _safe_float(row['val_loss']),
                'val/acc': _safe_float(row['val_acc']),
                'val/loss_std': _safe_float(row['val_loss_std']),
                'val/loss_min': _safe_float(row['val_loss_min']),
                'val/loss_max': _safe_float(row['val_loss_max']),
                'val/acc_std': _safe_float(row['val_acc_std']),
                'val/acc_min': _safe_float(row['val_acc_min']),
                'val/acc_max': _safe_float(row['val_acc_max']),
                'val/loss_mini_mean': _safe_float(row['val_loss_mini_mean']),
                'val/acc_mini_mean': _safe_float(row['val_acc_mini_mean']),
                'eval/checks_per_epoch': _safe_float(row['val_checks_per_epoch']),
                'eval/mini_eval_batches_per_check': _safe_float(row['mini_eval_batches_per_check']),
                'train/grad_norm': _safe_float(row['grad_norm']),
                'val/lr': _safe_float(row['lr']),
                'reliability/non_finite_loss_count': _safe_float(row['non_finite_loss_count']),
                'perf/samples_per_sec': _safe_float(row['samples_per_sec']),
                'perf/tokens_per_sec': _safe_float(row['tokens_per_sec']),
                'time/epoch_sec': _safe_float(row['epoch_sec']),
                'system/peak_gpu_mem_gb': _safe_float(row['peak_gpu_mem_gb']),
                'train/routing_entropy': _safe_float(row['routing_entropy']),
                'train/fusion_alpha': _safe_float(row['fusion_alpha']),
                'train/fusion_alpha_delta': _safe_float(row['fusion_alpha_delta']),
                'train/fusion_alpha_grad_abs_mean': _safe_float(row['fusion_alpha_grad_abs_mean']),
                'perf/flops_per_token_est': _safe_float(row['flops_per_token_est']),
                'perf/flops_per_step_est': _safe_float(row['flops_per_step_est']),
                'perf/flops_per_epoch_est': _safe_float(row['flops_per_epoch_est']),
                'energy/gpu_wh': _safe_float(row['energy_gpu_wh']),
                'energy/gpu_avg_power_w': _safe_float(row['gpu_avg_power_w']),
            }, step=epoch)

    total_train_sec = max(time.perf_counter() - train_start, 0.0)
    if wandb_run is not None:
        wandb_run.summary['time/total_train_sec'] = float(total_train_sec)
        wandb_run.summary['perf/flops_total_est'] = float(cumulative_flops_est)
        wandb_run.summary['perf/trainable_params'] = float(trainable_params)
        wandb_run.summary['energy/gpu_wh_total'] = float(cumulative_energy_gpu_wh)
        wandb_run.summary['eval/checks_per_epoch'] = float(eval_checks)
        wandb_run.summary['eval/mini_eval_batches_per_check'] = float(mini_eval_batches)
        wandb_run.summary['train/fusion_alpha_in_optimizer'] = float(alpha_in_optimizer)
        if history:
            wandb_run.summary['summary.val/acc'] = _safe_float(history[-1]['val_acc'])

    return history

In [ ]:
# Optional: print validation examples at each full-epoch eval
CFG.print_val_examples_each_epoch = True
CFG.val_examples_show_correct = 1  # desired number of correct examples per full eval
CFG.val_examples_show_incorrect = 1  # desired number of incorrect examples per full eval
CFG.val_example_char_limit = 220

_label_names = {
    0: 'contradiction/false',
    1: 'entailment/true',
    2: 'neutral/unknown',
}

_original_evaluate_model = evaluate_model
_full_eval_call_idx = 0

@torch.no_grad()
def evaluate_model(model: torch.nn.Module, loader: DataLoader, max_batches: int | None = None) -> dict:
    """Wrap original evaluate_model and optionally print validation predictions.

    Prints examples only for full validation passes (max_batches is None), which happen
    at epoch 0 and after each epoch in run_training.
    """
    global _full_eval_call_idx
    stats = _original_evaluate_model(model, loader, max_batches=max_batches)

    should_print = (
        max_batches is None
        and bool(getattr(CFG, 'print_val_examples_each_epoch', False))
    )
    if not should_print:
        return stats

    _full_eval_call_idx += 1
    want_correct = max(0, int(getattr(CFG, 'val_examples_show_correct', 1)))
    want_incorrect = max(0, int(getattr(CFG, 'val_examples_show_incorrect', 1)))
    char_limit = int(getattr(CFG, 'val_example_char_limit', 220))

    total_wanted = want_correct + want_incorrect
    if total_wanted == 0:
        return stats

    print(
        f"\n[Validation examples] full-eval #{_full_eval_call_idx} "
        f"(target: {want_correct} correct, {want_incorrect} incorrect)"
    )

    shown_correct = 0
    shown_incorrect = 0
    shown_total = 0

    # Scan minimally and run single-sample forward passes to avoid eval memory spikes.
    scanned_batches = 0
    for batch in loader:
        scanned_batches += 1
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        bsz = int(input_ids.size(0))
        for i in range(bsz):
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=torch.cuda.is_available()):
                out = model(
                    input_ids=input_ids[i:i + 1],
                    attention_mask=attention_mask[i:i + 1],
                )
                p = int(out.logits.argmax(dim=-1).item())

            y = int(labels[i].item())
            is_correct = (y == p)

            if is_correct and shown_correct >= want_correct:
                continue
            if (not is_correct) and shown_incorrect >= want_incorrect:
                continue

            text = tokenizer.decode(input_ids[i].detach().cpu().tolist(), skip_special_tokens=True)
            text = ' '.join(text.split())
            if char_limit > 0 and len(text) > char_limit:
                text = text[:char_limit] + '...'

            ok = 'OK' if is_correct else 'ERR'
            y_name = _label_names.get(y, str(y))
            p_name = _label_names.get(p, str(p))

            shown_total += 1
            print(f"[{shown_total}] {ok} gold={y_name} pred={p_name}")
            print(f"text: {text}")

            if is_correct:
                shown_correct += 1
            else:
                shown_incorrect += 1

            if shown_correct >= want_correct and shown_incorrect >= want_incorrect:
                break

        if shown_correct >= want_correct and shown_incorrect >= want_incorrect:
            break

    # Fallback: if one category was not found, print a note so behavior is explicit.
    if shown_correct < want_correct or shown_incorrect < want_incorrect:
        print(
            f"(Found {shown_correct}/{want_correct} correct and "
            f"{shown_incorrect}/{want_incorrect} incorrect examples in {scanned_batches} scanned validation batches.)"
        )

    return stats

## Train

In [ ]:
# Run one selected model across multiple seeds for K epochs (sequentially)
K_EPOCHS = CFG.k_epochs
base_model_name = selected_model_name
seeds_to_run = [int(s) for s in CFG.run_seeds] if CFG.run_seeds else [int(CFG.seed)]
seeds_to_run = list(dict.fromkeys(seeds_to_run))  # keep order, remove duplicates

print('Running model:', base_model_name)
print('Gate comparison axis:', CFG.gate_window_axis)
print('Learn fusion alpha:', CFG.learn_fusion_alpha)
print('Seeds:', seeds_to_run)

# ProofWriter branch uses train_collate/val_collate names; ensure run_training's mini-val path
# has a collate_fn symbol available.
if 'collate_fn' not in globals():
    collate_fn = getattr(val_loader, 'collate_fn', None)
    if collate_fn is None:
        raise RuntimeError('Could not resolve collate_fn from val_loader.')

history = []
param_count = None
run_name = base_model_name

for run_seed in seeds_to_run:
    print(f'\n=== Seed {run_seed} ===')
    set_seed(run_seed)

    run_name, model = build_model(base_model_name)
    baseline_is_param_matched = bool(
        run_name == 'baseline_llama' and CFG.use_param_matched_baseline
    )
    no_gate_uses_cross_attn = False if baseline_is_param_matched else None

    # Validate the effective fusion-alpha trainability against notebook config.
    effective_alpha_trainable = _fusion_alpha_trainable(model)
    effective_alpha_value = _extract_fusion_alpha(model)
    if effective_alpha_trainable is not None and bool(effective_alpha_trainable) != bool(CFG.learn_fusion_alpha):
        raise RuntimeError(
            f'Fusion alpha trainability mismatch: CFG.learn_fusion_alpha={CFG.learn_fusion_alpha}, '
            f'but model has alpha.requires_grad={effective_alpha_trainable}. '
            'Re-run Cells 6, 9, and 12 to refresh notebook state.'
        )
    print(
        'Fusion alpha effective state:',
        {'trainable': effective_alpha_trainable, 'value': effective_alpha_value, 'cfg_learn_fusion_alpha': CFG.learn_fusion_alpha}
    )

    if param_count is None:
        param_count = int(sum(p.numel() for p in model.parameters()))
        print('Params:', param_count)

    run_display_name = f'{run_name}-seed{run_seed}'
    run = wandb.init(
        project=CFG.wandb_project,
        group=CFG.wandb_group,
        name=run_display_name,
        reinit=True,
        config={
            'wandb': {
                'group': CFG.wandb_group,
                'run_name': run_display_name,
            },
            'model': {
                'name': CFG.model_name,
                'variant': run_name,
                'variant_note': 'baseline_param_matched' if baseline_is_param_matched else 'standard',
                'gate_mode': CFG.gate_mode,
                'gate_window_axis': CFG.gate_window_axis,
                'routing_top_k': None,
                'use_no_gate_stream': baseline_is_param_matched,
                'use_param_matched_baseline': baseline_is_param_matched,
                'param_matched_no_gate_update_type': CFG.param_matched_no_gate_update_type if baseline_is_param_matched else None,
                'no_gate_uses_cross_attn': no_gate_uses_cross_attn,
                'fusion_mode': 'mlp' if baseline_is_param_matched else 'linear_bridge',
                'alpha_init': CFG.alpha_init,
                'learn_fusion_alpha': CFG.learn_fusion_alpha,
                'fusion_alpha_effective_trainable': effective_alpha_trainable,
                'fusion_alpha_effective_value': effective_alpha_value,
                'lora': {'enabled': False},
                'logic_dim': CFG.logic_dim,
                'num_gates': CFG.num_gates,
                'cross_attn_heads': CFG.cross_attn_heads,
                'capacity_window_axis': CFG.gate_window_axis,
            },
            'train': {
                'epochs': CFG.k_epochs,
                'batch_size': CFG.train_batch_size,
                'eval_batch_size': CFG.eval_batch_size,
                'learning_rate': CFG.learning_rate,
                'weight_decay': CFG.weight_decay,
                'seed': run_seed,
                'max_length': CFG.max_length,
                'use_bf16_if_available': CFG.use_bf16_if_available,
            },
            'data': {
                'proofwriter_config': CFG.proofwriter_config,
                'proofwriter_depth': CFG.proofwriter_depth,
                'train_max_samples': CFG.train_max_samples,
                'val_max_samples': CFG.val_max_samples,
            },
        },
    )

    try:
        seed_history = run_training(model, name=run_name, epochs=K_EPOCHS, wandb_run=run)
        for row in seed_history:
            row['seed'] = run_seed
            row['model'] = run_name
        history.extend(seed_history)
    finally:
        run.finish()
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print('\nCompleted seeds:', seeds_to_run)

In [ ]:
# Show multi-seed results + plot val_acc / num_parameters
import matplotlib.pyplot as plt
import pandas as pd

results = pd.DataFrame(history).copy()
if 'model' not in results.columns:
    results['model'] = run_name
if 'seed' not in results.columns:
    results['seed'] = int(CFG.seed)

results['num_parameters'] = int(param_count)
results['val_acc_per_param'] = results['val_acc'] / results['num_parameters']

plot_df = results[results['epoch'] > 0].dropna(subset=['val_acc_per_param']).copy()
if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))

    for seed_value, g in plot_df.groupby('seed', sort=True):
        ax.plot(
            g['epoch'],
            g['val_acc_per_param'],
            marker='o',
            alpha=0.35,
            label=f'seed {seed_value}',
        )

    mean_curve = plot_df.groupby('epoch', as_index=False)['val_acc_per_param'].mean()
    ax.plot(
        mean_curve['epoch'],
        mean_curve['val_acc_per_param'],
        color='black',
        linewidth=2.5,
        marker='o',
        label='seed mean',
    )

    ax.set_title('Validation Accuracy per Parameter (Per Seed + Mean)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('val_acc / num_parameters')
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

results = results[[
    'model',
    'seed',
    'epoch',
    'num_parameters',
    'val_acc_per_param',
    'train_loss',
    'val_loss',
    'val_acc',
    'val_loss_mini_mean',
    'val_acc_mini_mean',
    'val_loss_std',
    'val_loss_min',
    'val_loss_max',
    'val_acc_std',
    'val_acc_min',
    'val_acc_max',
    'val_checks_per_epoch',
    'mini_eval_batches_per_check',
    'routing_entropy',
    'fusion_alpha',
    'fusion_alpha_delta',
    'fusion_alpha_grad_abs_mean',
    'grad_norm',
    'lr',
    'non_finite_loss_count',
    'samples_per_sec',
    'tokens_per_sec',
    'epoch_sec',
    'peak_gpu_mem_gb',
    'energy_gpu_wh',
    'gpu_avg_power_w',
    'flops_per_token_est',
    'flops_per_step_est',
    'flops_per_epoch_est',
]]
results.sort_values(['seed', 'epoch']).reset_index(drop=True)

## Mechanism Visuals (Routing + Gate Families)

These plots mirror the quick introspection style from logic_hf.ipynb, but run directly in this V4 workflow.

What this section shows for one validation batch:
1. Layer-by-channel routing heatmap
2. AND / OR / NOT family channel means (if routing channels are grouped as 3G)
3. Cross-attention entropy per layer (logic model path)

In [ ]:
# Mechanism visuals on one batch (safe for both logic and baseline selections)
import matplotlib.pyplot as plt
import numpy as np

viz_name, viz_model = build_model(selected_model_name)
viz_model.eval()

viz_batch = next(iter(val_loader))
viz_input_ids = viz_batch['input_ids'].to(device)
viz_attention_mask = viz_batch['attention_mask'].to(device)

with torch.no_grad():
    viz_out = viz_model(input_ids=viz_input_ids, attention_mask=viz_attention_mask)

routing_history = getattr(viz_out, 'routing_history', None)
cross_attn_history = getattr(viz_out, 'cross_attn_history', None)

if not routing_history:
    print('No routing history available for this selected model; skipping mechanism visuals.')
else:
    # [num_layers, channels] by averaging over tokens for first batch item
    route_mat = torch.stack([rw[0].detach().float().mean(dim=0) for rw in routing_history]).cpu().numpy()

    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(route_mat, aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_title(f'Routing Weights Heatmap ({viz_name})')
    ax.set_xlabel('Routing channel index')
    ax.set_ylabel('Transformer layer')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

    # If channels are grouped as AND/OR/NOT banks (3G), show per-family channel bars.
    if route_mat.shape[1] >= 3 and route_mat.shape[1] % 3 == 0:
        g = route_mat.shape[1] // 3
        and_mean = route_mat[:, :g].mean(axis=0)
        or_mean = route_mat[:, g:2 * g].mean(axis=0)
        not_mean = route_mat[:, 2 * g:].mean(axis=0)

        fig, axes = plt.subplots(1, 3, figsize=(14, 3.2), sharey=True)
        for ax, vals, title, color in [
            (axes[0], and_mean, 'AND family mean routing', '#2f6db3'),
            (axes[1], or_mean, 'OR family mean routing', '#3ca370'),
            (axes[2], not_mean, 'NOT family mean routing', '#c44e52'),
        ]:
            ax.bar(np.arange(len(vals)), vals, color=color, edgecolor='black', linewidth=0.4)
            ax.set_title(title)
            ax.set_xlabel('Channel index within family')
            ax.grid(axis='y', alpha=0.25)
        axes[0].set_ylabel('Mean routing weight')
        plt.tight_layout()
        plt.show()
    else:
        print('Routing channels are not in a 3-family layout; skipped AND/OR/NOT family bars.')

    if cross_attn_history:
        # Normalized entropy trend per layer for quick attention sharpness diagnostics.
        layer_entropy = []
        for probs in cross_attn_history:
            p = probs[0].detach().float().clamp_min(1e-12)
            p = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-12)
            ent = -(p * p.log()).sum(dim=-1).mean().item()
            layer_entropy.append(ent)

        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(np.arange(len(layer_entropy)), layer_entropy, marker='o', color='black')
        ax.set_title('Cross-Attention Entropy by Layer')
        ax.set_xlabel('Layer')
        ax.set_ylabel('Entropy')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print('No cross-attention history for this run path (expected for no-gate baseline mode).')

del viz_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## What To Tune Next

If runtime is too slow on Colab:
1. reduce `train_max_samples` and `val_max_samples`
2. reduce `k_epochs`
3. lower `max_length`
4. keep batch size at 1

If memory is still too high for full unfrozen Llama-3.1-8B, move to a larger GPU runtime or temporarily switch to a smaller real checkpoint to validate workflow.